# Modelling 5: Further evaluation
**Input:** the preprocessed features and the state saved by notebook 1. This notebook takes a closer look at the
chosen model, and at the follow-up model that the demo uses. Nothing is chosen or tuned on the test split here.
1. **Learning curve:** would more training posts help?
2. **Confidence intervals and significance:** how much of the gap between setups could be chance?
3. **Calibration:** when the model says 80%, is it right about 80% of the time?
4. **Selective prediction:** how accurate is the model on the posts it is sure about?

In [ ]:
# Make the shared code in src/ importable, wherever the repository folder is
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src", "mvsa")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from mvsa.modeling import *
# Imported by name: mvsa.modeling already has a macro_f1 that takes probabilities
from mvsa.evaluation import (bootstrap_scores, interval, mcnemar, reliability_table, expected_calibration_error,
                             fit_temperature, apply_temperature, risk_coverage, stratified_subsample)

state = load_state("run_state")
PRED, best_fam, best_setup, CFG = (state[k] for k in ("PRED", "best_fam", "best_setup", "CFG"))
train, val, test = D["train"], D["val"], D["test"]
yte = test["y"]
print(f"Chosen model: {best_fam} + {best_setup}")

## 1. Learning curve
Notebook 3 kept the training set fixed and changed the regularisation. Here the settings stay at their tuned values
and the training set grows, from 10% to all of it. Each size is drawn 5 times (one per seed), keeping the class
shares. If the validation score is still climbing at 100%, more labelled posts would help, which points to variance.
If the two curves have met and flattened, the model is limited by bias, and only a better model or better features
would help.

In [ ]:
FRACTIONS = [0.5, 1.0] if QUICK else [0.1, 0.25, 0.5, 0.75, 1.0]   # tiny subsets break the smoke test's data
hp, w = CFG[best_fam]["hp"], CFG[best_fam]["late_w"]

def subset(d, idx):
    """The same split, restricted to some of its posts. Every per-post array is indexed the same way."""
    n = len(d["y"])
    return {k: v[idx] if getattr(v, "shape", (0,))[0] == n else v for k, v in d.items()}

rows = []
for frac in FRACTIONS:
    for seed in SEEDS:
        sub = subset(train, stratified_subsample(train["y"], frac, seed))
        m = {k: fit(best_fam, hp[k], features(sub, k), sub["y"], seed) for k in ("text", "image", "early")}
        late = lambda d: w * m["text"].predict_proba(d["text"]) + (1 - w) * m["image"].predict_proba(d["image"])
        early = lambda d: m["early"].predict_proba(features(d, "early"))
        for setup, predict in (("late", late), ("early", early)):
            # Training score on the posts it was trained on, validation score on the full validation split
            rows.append({"setup": setup, "fraction": frac, "train_posts": len(sub["y"]), "seed": seed,
                         "train_macro_f1": macro_f1(sub["y"], predict(sub)),
                         "val_macro_f1": macro_f1(val["y"], predict(val))})

learning = pd.DataFrame(rows)
learning.to_csv(os.path.join(OUT_DIR, "learning_curve.csv"), index=False)
print(learning.groupby(["setup", "train_posts"])[["train_macro_f1", "val_macro_f1"]].agg(["mean", "std"]).round(3).to_string())

## 2. Confidence intervals and significance
The five seeds change the models' randomness, but every seed is scored on the same 447 test posts. A different
sample of test posts would give different scores, and with this few posts that is the bigger source of
uncertainty. The bootstrap estimates it: draw 447 posts with replacement, score every model on them, and repeat
2,000 times. All models are scored on the same draws, so the difference between two of them is also known for every
draw (a paired bootstrap). Logistic regression gives the same model for every seed, so seed 0 stands for all five.

In [ ]:
# Test predictions of every setup of the chosen family (seed 0), saved by notebook 1, plus the baseline
SETUPS = ["baseline_tfidf_hist", "text_only", "image_only", "early", "early_meta", "late", "gated"]
Xb = lambda d: sparse.hstack([d["tfidf"], sparse.csr_matrix(d["hist"])]).tocsr()
base = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000).fit(Xb(train), train["y"])
preds = {"baseline_tfidf_hist": base.predict(Xb(test))}
preds.update({s: PRED[(best_fam, s, 0)] for s in SETUPS[1:]})

boot = bootstrap_scores(yte, preds, n_boot=2000, seed=0)
rows = []
for s in SETUPS:
    low, high = interval(boot[s])
    rows.append({"setup": s, "macro_f1": f1_score(yte, preds[s], average="macro"),
                 "macro_f1_low": low[0], "macro_f1_high": high[0],
                 "accuracy": accuracy_score(yte, preds[s]), "accuracy_low": low[1], "accuracy_high": high[1]})
intervals = pd.DataFrame(rows).set_index("setup")
intervals.to_csv(os.path.join(OUT_DIR, "bootstrap_intervals.csv"))
print(f"{best_fam}, test split, 95% bootstrap intervals:")
print(intervals.round(3).to_string())

In [ ]:
# Paired comparisons. An interval that doesn't include 0 means the gap is unlikely to be chance.
PAIRS = [(best_setup, "baseline_tfidf_hist"), ("late", "early"), ("early", "gated"), ("early_meta", "early")]
rows = []
for a, b in dict.fromkeys(PAIRS):   # drops a repeated pair if the chosen setup is one of the others
    diff = boot[a] - boot[b]
    low, high = interval(diff)
    only_a, only_b, p = mcnemar(yte, preds[a], preds[b])
    rows.append({"a": a, "b": b,
                 "macro_f1_gain": f1_score(yte, preds[a], average="macro") - f1_score(yte, preds[b], average="macro"),
                 "macro_f1_low": low[0], "macro_f1_high": high[0], "share_a_better": (diff[:, 0] > 0).mean(),
                 "accuracy_gain": accuracy_score(yte, preds[a]) - accuracy_score(yte, preds[b]),
                 "accuracy_low": low[1], "accuracy_high": high[1],
                 "a_only_right": only_a, "b_only_right": only_b, "mcnemar_p": p})
differences = pd.DataFrame(rows)
differences.to_csv(os.path.join(OUT_DIR, "bootstrap_differences.csv"), index=False)
print(differences.round(3).to_string(index=False))

In [ ]:
# Notebook 2 ran McNemar's test for early against gated fusion on seed 0 only. Here it runs on every seed.
rows = []
for fam in FAMILIES:
    for seed in SEEDS:
        only_early, only_gated, p = mcnemar(yte, PRED[(fam, "early", seed)], PRED[(fam, "gated", seed)])
        rows.append({"family": fam, "seed": seed, "early_only_right": only_early, "gated_only_right": only_gated,
                     "p_value": p})
mcn = pd.DataFrame(rows)
mcn.to_csv(os.path.join(OUT_DIR, "mcnemar_all_seeds.csv"), index=False)
print("p-values, early against gated fusion:")
print(mcn.pivot(index="family", columns="seed", values="p_value").round(3).to_string())
print("\nSeeds where early fusion is significantly better at 5%:")
print(mcn[(mcn.p_value < 0.05) & (mcn.early_only_right > mcn.gated_only_right)].groupby("family").size()
      .reindex(FAMILIES, fill_value=0).to_string())

## 3. Calibration
The demo shows a confidence level with each prediction, so its probabilities should mean what they say. A
reliability table sorts the test posts into 10 bins by the model's top probability and compares the average
probability in each bin with the share it got right. The expected calibration error (ECE) is the average gap,
weighted by the number of posts in each bin: 0 is perfect.

Temperature scaling is the usual fix. One number, `T`, is fitted on the validation split and divides the
log-probabilities: `T > 1` makes the model less sure, `T < 1` more sure. It never changes which class is predicted,
so accuracy and F1 stay the same. Two models are checked: the chosen model, and the follow-up model used by the demo.

In [ ]:
# The chosen model, retrained with seed 0
predictors, _ = build(best_fam, CFG[best_fam], 0)
probs = {"chosen": {s: predictors[best_setup](D[s]) for s in ("val", "test")}}

# The follow-up model, rebuilt the way scripts/train_final_model.py builds it: CLIP text, CLIP image and TF-IDF
# logistic regressions, their probabilities averaged, then the neutral probability scaled down
fu = json.load(open(os.path.join(root, "results", "metrics", "followup_selection.json")))["chosen"]
if fu["setup"] == "late_tfidf":
    fu_text = fit("logreg", {"C": fu["C"]}, train["text"], train["y"], 0)
    fu_image = fit("logreg", {"C": fu["C"]}, train["image"], train["y"], 0)
    fu_words = LogisticRegression(class_weight="balanced", max_iter=5000).fit(train["tfidf"], train["y"])

    def followup(d):
        p = (fu_text.predict_proba(d["text"]) + fu_image.predict_proba(d["image"]) + fu_words.predict_proba(d["tfidf"])) / 3
        p[:, LABELS.index("neutral")] *= fu["neutral"]
        return p / p.sum(1, keepdims=True)

    probs["followup"] = {s: followup(D[s]) for s in ("val", "test")}
# Should match the saved results (0.603 and 0.643 on the full data)
print("Test macro-F1:", {name: round(macro_f1(yte, p["test"]), 4) for name, p in probs.items()})

In [ ]:
rows, tables = [], []
for name, p in probs.items():
    T = fit_temperature(val["y"], p["val"])        # fitted on validation, judged on test
    for version, P in (("raw", p["test"]), ("temperature scaled", apply_temperature(p["test"], T))):
        rows.append({"model": name, "version": version, "temperature": T if version != "raw" else 1.0,
                     "ece": expected_calibration_error(yte, P), "mean_confidence": P.max(1).mean(),
                     "accuracy": accuracy_score(yte, P.argmax(1)),
                     "log_loss": -np.log(P[np.arange(len(yte)), yte] + 1e-12).mean()})
        tables.append(reliability_table(yte, P).assign(model=name, version=version))
calibration = pd.DataFrame(rows)
calibration.to_csv(os.path.join(OUT_DIR, "calibration.csv"), index=False)
pd.concat(tables).to_csv(os.path.join(OUT_DIR, "reliability_bins.csv"), index=False)
print(calibration.round(3).to_string(index=False))
if "followup" in probs:
    print("\nReliability table, follow-up model, raw:")
    print(tables[2].round(3).to_string(index=False))

## 4. Selective prediction
A model that can say "not sure" can pass those posts to a person. If it only answers when its top probability is
at least some threshold, coverage (the share of posts it answers) falls, and accuracy on the posts it keeps should
rise. The table marks the demo's two cut-offs: 0.45, below which a prediction is shown as low confidence, and 0.70,
above which it is high confidence.

In [ ]:
THRESHOLDS = np.round(np.arange(0.30, 0.951, 0.05), 2)
coverage = pd.concat([risk_coverage(yte, p["test"], THRESHOLDS).assign(model=name) for name, p in probs.items()],
                     ignore_index=True)
coverage.to_csv(os.path.join(OUT_DIR, "risk_coverage.csv"), index=False)
print(coverage[coverage.threshold.isin([0.30, 0.45, 0.70])].round(3).to_string(index=False))

The figures for this notebook are drawn from the saved CSV files by `scripts/make_figures.py`, and the findings are
summarised in `docs/results.md`.